# 综合分析图纸 / Integrated Atlas
横向A1，输出PDF与PNG。独立读取Drive中已保存成果，不依赖旧笔记本变量，也不覆盖原成果。
先运行设置，再运行绘图。全部数据来自真实输入，无假数据。若缺少成果文件则停止。
布局：最终plots索引＋五层筛选＋建筑覆盖／食品店／行人伤亡事故／年龄／哮喘／交通六图。图中英文适合课程汇报。
默认重现已讨论的五层探索规则；健康用于背景比较，稀疏交通数据不用于统一排除。
生成后必须检查标签与图例。如果图纸需要我直接检查修改，请将生成的PNG或PDF发回。


In [ ]:
%pip -q install geopandas pyarrow matplotlib
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import json, textwrap
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from IPython.display import display, Image

ROOT = Path("/content/drive/MyDrive/ARCH6131_Scale")
RAW, OUT = ROOT / "data/raw", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
source = OUT / "all_enriched_plots.geojson"
if not source.exists():
    raise FileNotFoundError("Missing saved analysis: " + str(source))
m = gpd.read_file(source)
if m.crs is None:
    raise ValueError("Saved analysis has no CRS")
m = m.to_crs(2263)
need = ["plot_id", "LandUse", "BldgClass", "building_coverage_pct",
        "age65_pct", "age65_pct_GEOID", "food_store_count_buffer",
        "ped_casualty_crash_count_buffer", "adult_asthma_pct", "crash_count_buffer"]
missing = [c for c in need if c not in m]
if missing:
    raise ValueError("Saved analysis is incomplete; missing: " + ", ".join(missing))
m["plot_id"] = m.plot_id.astype(str)
assert m.plot_id.is_unique
assert m.geometry.notna().all() and m.geometry.is_valid.all()
for c in ["building_coverage_pct", "age65_pct", "food_store_count_buffer",
          "ped_casualty_crash_count_buffer", "adult_asthma_pct", "crash_count_buffer"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")
tract_age = m[["age65_pct_GEOID", "age65_pct"]].dropna().drop_duplicates()
assert tract_age.age65_pct_GEOID.is_unique
thresholds = {
    "building": float(m.building_coverage_pct.quantile(.25)),
    "age": float(tract_age.age65_pct.median()),
    "food": float(m.food_store_count_buffer.median()),
    "ped": float(m.ped_casualty_crash_count_buffer.median())
}
assert all(np.isfinite(list(thresholds.values())))
lu = pd.to_numeric(m.LandUse, errors="coerce")
bc = m.BldgClass.astype("string").str.strip().str.upper()
vacant = lu.eq(11)
parking = bc.isin(["G6", "G7", "Z2", "RP"])
eligible = pd.Series(pd.NA, index=m.index, dtype="boolean")
known = lu.notna() & bc.notna() & bc.ne("")
eligible.loc[known] = False
eligible.loc[vacant | parking] = True
m["space_type_eligible"] = eligible
m["space_type"] = "Other / unverified"
m.loc[vacant, "space_type"] = "Vacant land"
m.loc[parking & ~vacant, "space_type"] = "Surface parking"

# Reproduce the agreed exploratory rules from the full baseline.
rules = [
    ("Low building coverage", "building_coverage_pct", "<=", thresholds["building"]),
    ("Higher older-adult share", "age65_pct", ">=", thresholds["age"]),
    ("Fewer nearby food retailers", "food_store_count_buffer", "<=", thresholds["food"]),
    ("More pedestrian-casualty crashes", "ped_casualty_crash_count_buffer", ">=", thresholds["ped"]),
    ("Vacant land / surface parking", "space_type_eligible", "==", True)
]
active = pd.Series(True, index=m.index)
m["selection_status"] = "active"
m["first_reason"] = ""
funnel = [{"stage": "Initial plots", "remaining": len(m), "review": 0, "excluded": 0}]
for name, col, op, value in rules:
    unknown = active & m[col].isna()
    passes = (m[col].le(value) if op == "<=" else
              m[col].ge(value) if op == ">=" else m[col].eq(value)).fillna(False)
    fail = active & m[col].notna() & ~passes
    m.loc[unknown, ["selection_status", "first_reason"]] = ["review", name + ": missing"]
    m.loc[fail, ["selection_status", "first_reason"]] = ["excluded", name]
    active = active & m[col].notna() & passes
    funnel.append({"stage": name, "remaining": int(active.sum()),
                   "review": int(unknown.sum()), "excluded": int(fail.sum())})
selected = m.loc[active].copy().sort_values("plot_id")
if selected.empty:
    raise ValueError("No plots pass the five rules. Review the funnel; do not tune solely to obtain 11 plots.")
selected["item_id"] = [f"P{i:02d}" for i in range(1, len(selected)+1)]
funnel = pd.DataFrame(funnel)
display(funnel)
print("Final plots:", len(selected))

boundary_file = RAW / "boundary.geojson"
boundary = gpd.read_file(boundary_file).to_crs(2263) if boundary_file.exists() else None
bounds = m.total_bounds
dx, dy = bounds[2]-bounds[0], bounds[3]-bounds[1]
xlim = (bounds[0]-.04*dx, bounds[2]+.04*dx)
ylim = (bounds[1]-.04*dy, bounds[3]+.04*dy)
navy, ink, muted, paper = "#142b3a", "#243b46", "#647782", "#f7f5ef"
orange, teal = "#d95c36", "#16877b"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 13,
                     "text.color": ink, "axes.labelcolor": ink,
                     "pdf.fonttype": 42})
fig = plt.figure(figsize=(33.11, 23.39), facecolor=paper)  # A1 landscape
fig.text(.035, .965, "SITING A COMMUNITY FOOD MARKET", fontsize=37,
         fontweight="bold", color=navy)
fig.text(.035, .940, "JACKSON HEIGHTS + CORONA  /  PLOT-BASED SCREENING ATLAS", fontsize=18)
argument = ("Food-market siting should consider residents' needs, pedestrian safety and spatial opportunity together. "
            "Layered screening identifies vacant plots and surface parking for further assessment; "
            "a neighborhood fish market is a proposed use, not a demonstrated outcome.")
fig.text(.035, .915, textwrap.fill(argument, 155), fontsize=14, linespacing=1.5)
gs = fig.add_gridspec(3, 3, left=.035, right=.965, bottom=.105, top=.865,
                      height_ratios=[1.05, 1, 1], hspace=.48, wspace=.24)

def base(ax):
    ax.set_facecolor(paper)
    m.plot(ax=ax, facecolor="#e2e3dd", edgecolor="none", rasterized=True)
    if boundary is not None:
        boundary.boundary.plot(ax=ax, color="#9ba8aa", linewidth=.55)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_axis_off()

def heading(ax, number, title, note):
    ax.text(0, 1.10, number + "  " + title, transform=ax.transAxes,
            fontsize=18, fontweight="bold", color=navy)
    ax.text(0, 1.025, note, transform=ax.transAxes, fontsize=11.5, color=muted)

def colorbar(ax, cmap, vmin, vmax, label):
    sm = plt.cm.ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal",
                     fraction=.038, pad=.015, aspect=45)
    cb.set_label(label, fontsize=11)
    cb.ax.tick_params(labelsize=10)

ax = fig.add_subplot(gs[0, :2])
base(ax)
for kind, color in [("Vacant land", orange), ("Surface parking", teal)]:
    subset = selected.loc[selected.space_type.eq(kind)]
    if not subset.empty:
        subset.plot(ax=ax, facecolor=color, edgecolor=color, linewidth=.7)
        # Pins keep small plots visible; true plot polygons remain underneath.
        points = subset.geometry.representative_point()
        ax.scatter(points.x, points.y, s=30, color=color, zorder=5)
        if len(selected) <= 40:
            for (_, row), point in zip(subset.iterrows(), points):
                ax.annotate(row.item_id, (point.x, point.y), xytext=(4,4),
                            textcoords="offset points", fontsize=9, color=navy)
heading(ax, "00", "FINAL CANDIDATE INDEX",
        f"{len(m):,} initial plots  /  {len(selected):,} candidates  /  pins identify small polygons")
ax.legend(handles=[
    Line2D([0],[0], marker="o", color="none", markerfacecolor=orange, label="Vacant land"),
    Line2D([0],[0], marker="o", color="none", markerfacecolor=teal, label="Surface parking")],
    loc="lower left", frameon=True, fontsize=11)
# Graphic scale in projected feet; arrow indicates grid north.
sx, sy = xlim[0]+.07*(xlim[1]-xlim[0]), ylim[0]+.10*(ylim[1]-ylim[0])
ax.plot([sx,sx+1000],[sy,sy], color=navy, linewidth=3)
ax.text(sx,sy+.018*dy,"1,000 ft",fontsize=10)
ax.annotate("Grid N", xy=(.95,.90), xytext=(.95,.75),
            xycoords="axes fraction", ha="center",
            arrowprops={"arrowstyle":"->","color":navy}, fontsize=11)

ax = fig.add_subplot(gs[0, 2]); ax.set_axis_off()
ax.text(0, 1.10, "METHOD  /  FIVE-LAYER SCREEN", transform=ax.transAxes,
        fontsize=18, fontweight="bold", color=navy)
details = [
    "Baseline: all intersecting MapPLUTO plots",
    f"Building coverage <= {thresholds['building']:.2f}% (baseline Q25)",
    f"Age 65+ >= {thresholds['age']:.2f}% (unique-tract median)",
    f"Food retailers <= {thresholds['food']:g} (baseline median)",
    f"Pedestrian-casualty crashes >= {thresholds['ped']:g} (baseline median)",
    "LandUse 11 or BldgClass G6 / G7 / Z2 / RP"
]
for i, (row, detail) in enumerate(zip(funnel.to_dict("records"), details)):
    y = .94-i*.125
    ax.text(0,y,f"{i:02d}  {row['stage']}",fontsize=13,fontweight="bold")
    ax.text(1,y,f"{row['remaining']:,}",ha="right",fontsize=18,color=orange,fontweight="bold")
    ax.text(0,y-.044,detail,fontsize=10.5,color=muted)
ax.text(0,.12, f"Missing-data review: {int((m.selection_status=='review').sum())} plots. "
        "Thresholds fixed before sequential filtering.", fontsize=11, wrap=True)
ax.text(0,.01, textwrap.fill("Exploratory thresholds, not planning standards. "
        "Low footprint coverage does not establish vacancy or permission to redevelop. "
        "Health is a contextual comparison, not an exclusion rule.", 65), fontsize=11, color=muted)

def plot_metric(slot, number, title, field, cmap, note, label):
    ax = fig.add_subplot(slot); base(ax)
    valid = m[field].dropna()
    if valid.empty:
        ax.text(.5,.5,"No valid data",transform=ax.transAxes,ha="center")
    else:
        lo, hi = float(valid.min()), float(valid.max())
        hi = hi if hi > lo else lo+1
        m.loc[m[field].notna()].plot(ax=ax,column=field,cmap=cmap,vmin=lo,vmax=hi,
                                     edgecolor="none",rasterized=True)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        colorbar(ax,cmap,lo,hi,label)
    heading(ax,number,title,note)

plot_metric(gs[1,0],"01","BUILDING COVERAGE","building_coverage_pct","YlGnBu",
            "Building / plot intersection; overlapping footprints unioned","Mapped building coverage (%)")
plot_metric(gs[1,1],"02","FOOD RETAIL ACCESS CONTEXT","food_store_count_buffer","YlOrBr",
            "Unique mapped food-retail licences within a 660-ft plot buffer","Food-retail licence count")
plot_metric(gs[1,2],"03","PEDESTRIAN CASUALTY CRASHES","ped_casualty_crash_count_buffer","OrRd",
            "2021-2025; crashes with pedestrian injury/death within 660 ft","Crash events (five-year total)")

def tract_panel(slot, number, title, filename, field, cmap, note):
    ax = fig.add_subplot(slot); base(ax)
    path = RAW / filename
    if not path.exists():
        raise FileNotFoundError("Required spatial context: " + str(path))
    t = gpd.read_file(path).to_crs(2263)
    t[field] = pd.to_numeric(t[field],errors="coerce")
    # Clip only the display geometry, not the source statistical units.
    display_area = boundary.geometry.union_all() if boundary is not None else m.geometry.union_all()
    t = gpd.clip(t, display_area)
    valid = t[field].dropna()
    if valid.empty:
        raise ValueError(filename + " has no valid values in study area")
    lo,hi = float(valid.min()),float(valid.max())
    hi = hi if hi>lo else lo+1
    t.loc[t[field].notna()].plot(ax=ax,column=field,cmap=cmap,vmin=lo,vmax=hi,
                                edgecolor=paper,linewidth=.45,rasterized=True)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    colorbar(ax,cmap,lo,hi,"Tract estimate (%)")
    heading(ax,number,title,note)

tract_panel(gs[2,0],"04","OLDER-ADULT POPULATION","age_tracts.geojson","age65_pct","BuPu",
            "ACS 2020-2024; age 65+ share; shown at census-tract scale")
tract_panel(gs[2,1],"05","ADULT CURRENT ASTHMA","health_tracts.geojson","adult_asthma_pct","PuRd",
            "CDC PLACES crude model estimate; adult / tract-level context")

ax = fig.add_subplot(gs[2,2]); base(ax)
traffic_path = OUT / "traffic_2024_sites_provisional.geojson"
traffic_note = "No saved 2024 traffic-site summary; traffic layer unavailable."
if traffic_path.exists():
    t = gpd.read_file(traffic_path)
    if t.crs is None: raise ValueError("Traffic file has no CRS")
    t = t.to_crs(2263)
    field = "weekday_08_09_mean_vehicles"
    t[field] = pd.to_numeric(t[field],errors="coerce")
    t = t.loc[t[field].notna()].copy()
    if not t.empty:
        lo,hi=float(t[field].min()),float(t[field].max())
        hi=hi if hi>lo else lo+1
        t.plot(ax=ax,column=field,cmap="viridis",vmin=lo,vmax=hi,
               markersize=150,edgecolor=navy,linewidth=.8)
        label_rows = []
        for _,r in t.iterrows():
            label_rows.append(str(r.get("street","")) + " " + str(r.get("direction","")) +
                              ": " + f"{r[field]:.1f}" + " veh/h; n=" + str(r.get("observed_dates","?")))
        ax.text(0,.015,"\n".join(label_rows),transform=ax.transAxes,fontsize=9,
                bbox={"facecolor":paper,"alpha":.92,"edgecolor":"none"})
        colorbar(ax,"viridis",lo,hi,"Directional mean observed vehicles / hour")
        traffic_note = "2024 Mon-Fri, 08:00-09:00; 2-5 dates; directions share locations"
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
heading(ax,"06","OBSERVED TRAFFIC VOLUMES",traffic_note)
ax.text(0,-.23,"Sparse sampling; point CRS interpreted as EPSG:2263 remains provisional.\n"
        "No traffic value assigned to unobserved plots.",transform=ax.transAxes,
        fontsize=10,color=muted)

footer = [
    "SOURCES  MapPLUTO 26v2 | BUILDING_20261026 | NYS Retail Food Stores_20261102 | NYPD Motor Vehicle Collisions | ACS 2024 5-year B01001 | CDC PLACES | NYC DOT ATR",
    "METHOD  660-ft Euclidean buffers from plot edges; overlapping buffers can share events. Counts are not exposure-adjusted risk or network accessibility. "
    "Large plots cover larger search areas.",
    "LIMITS  Food licences are not seafood demand. Tract values are not plot-level observations; do not sum repeated tract populations. "
    "Asthma is crude modelled adult prevalence, not evidence of traffic causation.",
    "CHECK  Missing coordinates, statistical uncertainty, current site use, ownership and feasibility require review. "
    "PLACES release/indicator year must be checked against download metadata; file dates are not observation dates."
]
for i,line in enumerate(footer):
    fig.text(.035,.070-i*.014,line,fontsize=10.3,color=muted)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
prefix = OUT / ("Scale_Integrated_Atlas_" + stamp)
png_path, pdf_path = Path(str(prefix)+".png"), Path(str(prefix)+".pdf")
fig.savefig(png_path,dpi=180,facecolor=paper)
fig.savefig(pdf_path,facecolor=paper)
selected.to_crs(4326).to_file(str(prefix)+"_selected.geojson",driver="GeoJSON")
selected.drop(columns="geometry").to_csv(str(prefix)+"_items.csv",index=False)
funnel.to_csv(str(prefix)+"_funnel.csv",index=False)
m.drop(columns="geometry").to_csv(str(prefix)+"_audit.csv",index=False)
Path(str(prefix)+"_rules.json").write_text(json.dumps(
    {"thresholds":thresholds,"rules":rules,"buffer_feet":660,
     "age_threshold_basis":"unique tracts represented by initial plots",
     "note":"exploratory; parking/vacant rule selected by user"},
    indent=2),encoding="utf-8")
plt.close(fig)
display(Image(filename=str(png_path),width=1500))
print("A1 PDF:",pdf_path)
print("PNG:",png_path)
print("Item IDs / funnel / audit / rules saved alongside.")
if len(selected)<11:
    print("WARNING: fewer than 11 Items. Review data/method, do not tune only to meet count.")
print("Please inspect labels, maps and footer before submission; rendering has not been visually reviewed by the assistant.")
